# 02 — Kinematics

Inspect original SAM outputs and plot **independently calculated** left/right joint angles. The notebook does not perform synchronization or Vicon calibration.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Start Jupyter in markerless_sam/ or notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "lunge_sam_out"
RESULTS = ROOT / "results"
print("Repository:", ROOT, "| CSV folder:", DATA.exists())

In [ ]:
SUBJECT, TRIAL, CAMERA = "S11", "lunge_01", "cam0"
RAW = DATA / SUBJECT / TRIAL / CAMERA / "sam_raw.npz" # this is the raw SAM output, which is used for validation
print("SAM raw NPZ exists:", RAW.exists())

In [ ]:
if RAW.exists():
    with np.load(RAW, allow_pickle=False) as z:
        print("Available arrays:")
        for key in z.files:
            print(key, z[key].shape, z[key].dtype)
        if {"frame", "time_ms", "keypoints3d"} <= set(z.files):
            frames, t, points = z["frame"], z["time_ms"], z["keypoints3d"]
            assert len(frames) == len(t) == len(points)
            assert np.all(np.diff(frames) > 0)
            assert np.all(np.diff(t) > 0)
            print("Approximate effective sampling rate:", 1000/np.median(np.diff(t)), "Hz")
else:
    print("First transfer sam_raw.npz from the inference server.")

In [ ]:
angle_files = sorted(DATA.glob("*joint_angles.csv"))
print("Processed angle files:", len(angle_files))
for path in angle_files[:10]:
    print(path.name, list(pd.read_csv(path, nrows=0).columns))

### Implementation next
Transfer the server geometric joint-plane calculations and MHR parent-relative rotation decomposition into tested functions. Standardize hip/knee/ankle sign, coordinate frame and anatomical definitions before comparing to Vicon. Do not select Euler axes using the evaluation trials.

## Bilateral SAM visualization

This reads **original-timebase** SAM angle exports only. Left = red; right = green. Missing left-side columns are reported rather than substituted by right-side values.


In [ ]:
# Bilateral plotting from original-timebase SAM angle CSVs, when available
import importlib.util
module_path = ROOT / "src" / "kinematics.py"
if not module_path.exists():
    raise FileNotFoundError(f"Install the updated module at {module_path}")
from src.kinematics import plot_bilateral_angles

SUBJECT, TRIAL = "S9", "lunge_01"
CAMERAS = ("cam0", "cam1")
METHOD = "geo"
for camera in CAMERAS:
    candidates = [
        DATA / f"{SUBJECT.lower()}_lunge_{camera}_sam_angles_raw.csv",
        DATA / SUBJECT / TRIAL / camera / "sam_angles_raw.csv",
    ]
    csv_path = next((p for p in candidates if p.exists()), None)
    if csv_path is None:
        print(f"{camera}: no original-timebase SAM angle CSV found (export required)")
        continue
    df = pd.read_csv(csv_path)
    if "time_ms_aligned" in df.columns:
        print(f"{camera}: SKIPPED — contains time_ms_aligned (pre-synchronized export)")
        continue
    if "time_ms" not in df.columns:
        print(f"{camera}: SKIPPED — missing original time_ms")
        continue
    fig, axes, missing = plot_bilateral_angles(df, method=METHOD,
        title=f"{SUBJECT} {TRIAL} {camera} — original video time")
    display(fig)
    import matplotlib.pyplot as plt
    plt.close(fig)
    if missing:
        print(f"{camera}: missing angle columns: {missing}")
